# THE FINAL GPU RUN - everything that needs a GPU, in one pass

Runtime -> Run all, then leave it. Expect **3-5 hours**.

Three jobs, in dependency order (the judge must go last - it scores whatever
generation has produced):

| job | what | GPU | ~time |
|---|---|---|---|
| **A** | full-A/D causal ablation, 4 branches | yes | ~1.5 h |
| **B** | cross-fitted causal ablation, 4 branches | yes | ~40 min |
| **C** | re-judge everything + recompute CF1/CF2 | yes | ~1.5-2.5 h |

After this, **nothing else needs a GPU.** What's left is CPU-only (the n=150
McNemar, the D-source and factorial tables) plus writing.

### If Colab disconnects
Just re-run all. Jobs A and B are shard-resumable - completed shards are not
regenerated. Only job C (the judge) restarts from scratch, which is why it
runs last.

### What this deliberately does NOT touch
The frozen held-out CF2 files (`causal_ablation_v2_{stage}_L24-28.json`).
Job A writes `_fullAD.json`, job B writes `_xfit5.json`. Separate files,
separate shard units, separate condition names.


## 0. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Clone + pin + install

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/urosavurdic/dpo-safety-representations.git'
REPO_DIR = '/content/dpo-safety-representations'
BRANCH = 'agent/c-quadrant-end-to-end-e0e2317a'
PINNED_COMMIT = '91632e5b46e5e2e357d054ed2e08bd1aff52ac53'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(['git', 'fetch', 'origin'], check=True)
subprocess.run(['git', 'checkout', PINNED_COMMIT], check=True)
print('checked out', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
!pip -q install -r requirements.txt
!pip -q install -U "bitsandbytes>=0.46.1"
!pip uninstall -y torchao || true
!nvidia-smi


## 2. Bind results/ to Drive

Auto-detects the real `dpo_v2` folder (a stale empty `MyDrive/dpo_v2` stub
would otherwise shadow the shared one). HF cache stays OFF Drive - the judges
are ~20 GB and the quota guard cannot see the real Drive quota.

In [ ]:
import os, glob

candidates = ['/content/drive/MyDrive/dpo_v2']
candidates += sorted(glob.glob('/content/drive/.shortcut-targets-by-id/*/dpo_v2'))
candidates += sorted(glob.glob('/content/drive/Shareddrives/*/dpo_v2'))
real_root = None
for c in candidates:
    if glob.glob(os.path.join(c, 'results', 'activations', '*_final.npy')):
        real_root = c; break
if real_root is None:
    raise SystemExit('No dpo_v2 folder with real activations found. Checked:\n  '
                     + '\n  '.join(candidates))
os.environ['DPO_DRIVE_ROOT'] = real_root
print('DPO_DRIVE_ROOT =', real_root)

from src.colab_persist import bind, status_line
info = bind(persist_hf_cache=False)
print(status_line(info))


## 3. HuggingFace auth - REQUIRED for job C

Colab secret `HF_TOKEN` (key icon, notebook access ON). The account must have
accepted the `google/gemma-2b` **and** `allenai/wildguard` licences.

Jobs A and B do not need this. Job C does, so this cell fails loudly rather
than letting you discover it 2 hours in.

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login

_tok = userdata.get('HF_TOKEN')            # the SECRET NAME, not the token
os.environ['HF_TOKEN'] = _tok              # env var: inherited by !python subprocesses
login(token=_tok)
print('HF login OK - job C can run')


## 4. PREFLIGHT - fails before spending any GPU time

Checks the things that have silently produced wrong-but-plausible output in
this project before: stale 370-row activation metadata, a missing `split`
key, a stage that was never re-extracted.

In [ ]:
import json, glob, os, sys

STAGES = ['M3', 'M3_direct', 'M3_alt', 'M3_direct_alt']
problems = []

bench = sorted(glob.glob('data/frozen_v2/benchmark_v2_*.jsonl'))[-1]
rows = [json.loads(l) for l in open(bench, encoding='utf-8') if l.strip()]
from collections import Counter
q = Counter(r['quadrant'] for r in rows)
print(f'benchmark: {os.path.basename(bench)}  n={len(rows)}  {dict(q)}')
if len(rows) != 654:
    problems.append(f'benchmark has {len(rows)} rows, expected 654')

ad = [r for r in rows if r['quadrant'] in ('A', 'D')]
sp = Counter(r.get('split') for r in ad)
print(f'A/D splits: {dict(sp)}')
if sp.get('direction_estimation') != 240 or sp.get('held_out_behavioral') != 60:
    problems.append(f'unexpected split sizes {dict(sp)} (expected 240/60)')

n_a_est = sum(1 for r in rows if r['quadrant'] == 'A'
              and r.get('split') == 'direction_estimation')
print(f'quadrant-A estimation rows (what job B cross-fits): {n_a_est}')

print('\nactivation metadata:')
for st in STAGES:
    p = f'results/activations/{st}_metadata.json'
    if not os.path.exists(p):
        problems.append(f'{st}: no activation metadata'); print(f'  {st:16s} MISSING'); continue
    m = json.load(open(p, encoding='utf-8', errors='replace'))
    with_split = sum(1 for r in m if r.get('split'))
    print(f'  {st:16s} rows={len(m):<5d} with_split={with_split}')
    if len(m) != 654:
        problems.append(f'{st}: {len(m)} rows, expected 654 (STALE - re-extract before running)')
    if with_split != 300:
        problems.append(f'{st}: {with_split} rows carry a split, expected 300')

print('\nheld-out causal files that must NOT be modified:')
for st in STAGES:
    p = f'results/raw/causal_ablation_v2_{st}_L24-28.json'
    print(f'  {st:16s} {"present" if os.path.exists(p) else "ABSENT"}')

if problems:
    print('\n' + '='*70)
    for p in problems:
        print('  BLOCKED:', p)
    raise SystemExit('Preflight failed - fix the above before spending GPU time.')
print('\nPREFLIGHT PASSED - jobs A, B, C are safe to run.')


---
# JOB A - full-A/D causal ablation (~1.5 h)

Quadrants A and D in full (150 each, direction-estimation half included),
B/C skipped. This is the **predeclared sensitivity analysis**, not the
confirmatory endpoint - the held-out 30 stays the anchor.

The 30 held-out A rows get regenerated here as well as in the frozen file.
Greedy decoding makes that a repeat rather than a second draw; where the
merged judge output holds both, CF2's `primary` block reads one of two
responses that should be identical. Worth knowing, not worth special-casing.

M3 runs alone first with a hard row-count assert.

In [ ]:
!python -m src.analysis.v2_pipeline causal --stage M3 --all-ad-sensitivity


In [ ]:
import json, os
from collections import Counter

p = 'results/raw/causal_ablation_v2_M3_L24-28_fullAD.json'
assert os.path.exists(p), 'SMOKE TEST FAILED - file not written'
rows = json.load(open(p, encoding='utf-8', errors='replace'))
print('rows:', len(rows), '(expect 900 = 300 A/D prompts x 3 conditions)')
print('by condition:', dict(Counter(r.get('stage') for r in rows)))
print('by quadrant :', dict(Counter(r.get('quadrant') for r in rows)))
assert len(rows) == 900, f'expected 900 rows, got {len(rows)} - STOP and report'
assert os.path.exists('results/raw/causal_ablation_v2_M3_L24-28.json'), \
    'the frozen held-out file vanished - STOP'
print('\nJOB A SMOKE TEST PASSED')


In [ ]:
for st in ['M3_direct', 'M3_alt', 'M3_direct_alt']:
    get_ipython().system(f'python -m src.analysis.v2_pipeline causal --stage {st} --all-ad-sensitivity')


In [ ]:
import json, os
print('JOB A results:')
ok = True
for st in ['M3', 'M3_direct', 'M3_alt', 'M3_direct_alt']:
    p = f'results/raw/causal_ablation_v2_{st}_L24-28_fullAD.json'
    if not os.path.exists(p):
        print(f'  {st:16s} MISSING'); ok = False; continue
    n = len(json.load(open(p, encoding='utf-8', errors='replace')))
    print(f'  {st:16s} {n} rows {"OK" if n == 900 else "<-- UNEXPECTED"}')
    ok = ok and n == 900
print('JOB A', 'COMPLETE' if ok else 'INCOMPLETE - report before continuing')


---
# JOB B - cross-fitted causal ablation (~40 min)

The bias fix for the estimation half. Each of the 120 quadrant-A estimation
rows is generated under a direction estimated **without it** (K=5 folds), so
its ~1/120 self-influence on the centroid is removed.

Report the result as an **out-of-fold n=120 estimate**, never "independent
n=120" - the 5 training portions overlap by 3/4.

Writes `_xfit5.json` with conditions named `{stage}_xfit_*`, so cross-fitted
rows can never collide with job A's rows for the same prompt.

In [ ]:
!python -m src.analysis.v2_pipeline causal --stage M3 --cross-fit 5


In [ ]:
import json, os
from collections import Counter

p = 'results/raw/causal_ablation_v2_M3_L24-28_xfit5.json'
assert os.path.exists(p), 'SMOKE TEST FAILED - file not written'
rows = json.load(open(p, encoding='utf-8', errors='replace'))
print('rows:', len(rows), '(expect 360 = 120 A-estimation prompts x 3 conditions)')
print('by condition:', dict(Counter(r.get('stage') for r in rows)))
print('by fold     :', dict(Counter(r.get('xfit_fold') for r in rows)))
assert len(rows) == 360, f'expected 360 rows, got {len(rows)} - STOP and report'

side = json.load(open(p.replace('.json', '_binding.json'), encoding='utf-8'))
folds = side['folds']
flat = [rid for f in folds for rid in f['test_record_ids']]
assert len(flat) == len(set(flat)) == 120, \
    f'fold partition is not a clean cover: {len(flat)} entries, {len(set(flat))} distinct'
print('\nfolds (disjoint, covering all 120):')
for f in folds:
    print(f"  fold {f['fold']}: {f['n_test_rows']:2d} test rows, d from "
          f"{f['n_direction_A_rows']} A + {f['n_direction_D_rows']} D, "
          f"cos(d_fold, d_full)@L24 = {f['cos_with_full_direction_at_L24']:.4f}")
print('\nJOB B SMOKE TEST PASSED')


In [ ]:
for st in ['M3_direct', 'M3_alt', 'M3_direct_alt']:
    get_ipython().system(f'python -m src.analysis.v2_pipeline causal --stage {st} --cross-fit 5')


In [ ]:
import json, os
print('JOB B results:')
ok = True
for st in ['M3', 'M3_direct', 'M3_alt', 'M3_direct_alt']:
    p = f'results/raw/causal_ablation_v2_{st}_L24-28_xfit5.json'
    if not os.path.exists(p):
        print(f'  {st:16s} MISSING'); ok = False; continue
    n = len(json.load(open(p, encoding='utf-8', errors='replace')))
    side = json.load(open(p.replace('.json', '_binding.json'), encoding='utf-8'))
    cs = [f['cos_with_full_direction_at_L24'] for f in side['folds']]
    print(f'  {st:16s} {n} rows {"OK" if n == 360 else "<-- UNEXPECTED"}   '
          f'cos(d_fold,d_full)@L24 min={min(cs):.4f} max={max(cs):.4f}')
    ok = ok and n == 360
print('JOB B', 'COMPLETE' if ok else 'INCOMPLETE - report before continuing')


---
# JOB C - re-judge everything (~1.5-2.5 h)

Scores the new responses with StrongREJECT (continuous, primary) and
WildGuard (binary, secondary).

**The manifest cell below is a guard, not a formality.** The scanner's glob
used to be anchored to `_L24-28.json`, which silently excluded every
`_fullAD` / `_xfit` file - the judge would have run for two hours and
produced a file that looked complete while every new block stayed at n=0.
The assert catches that class of failure immediately.

In [ ]:
# Built in-process on purpose. The CLI's --from-results-dir builds the
# manifest AND immediately starts the 2-hour judge, which would put the guard
# below after the thing it is meant to guard.
import os
os.makedirs('results/manifests', exist_ok=True)
from src.analysis.behavioral_judges import build_consolidated_from_results
_ = build_consolidated_from_results('results', 'results/manifests/consolidated_judge.json')


In [ ]:
import json, os

mp = 'results/manifests/consolidated_judge.json'
man = json.load(open(mp, encoding='utf-8'))
files = [e['response_file'] for e in man['entries']]
print(f'{len(files)} response files in the manifest')

need = []
for st in ['M3', 'M3_direct', 'M3_alt', 'M3_direct_alt']:
    need += [f'causal_ablation_v2_{st}_L24-28_fullAD.json',
             f'causal_ablation_v2_{st}_L24-28_xfit5.json']
missing = [n for n in need if not any(f.endswith(n) for f in files)]
for n in need:
    print(('  OK      ' if n not in missing else '  MISSING ') + n)
assert not missing, (
    f'{len(missing)} new response files are NOT in the judge manifest - the '
    f'judge would silently skip them. STOP and report.')
print('\nmanifest covers every job A + job B output')

# how much work job C is actually about to do, before committing to it
from src.analysis.behavioral_judges import verify_manifest_entry, _row_in_scope
n_scope = 0
for e in man['entries']:
    try:
        rows = verify_manifest_entry(e, man['benchmark_sha256'],
                                     man['split_manifest_sha256'])
    except Exception as ex:
        print('  could not pre-count', e['response_file'], '-', ex)
        continue
    n_scope += sum(1 for r in rows if _row_in_scope(r, 'confirmatory'))
print(f'\n~{n_scope} rows will get model judging (StrongREJECT + WildGuard)')
print(f'   rough estimate at ~1.6 s/row: {n_scope * 1.6 / 3600:.1f} h')


### The long one. ~1.5-2.5 h. Not resumable - if it dies, re-run all
(jobs A and B will skip instantly from their shards).

In [ ]:
!python -m src.analysis.behavioral_judges \
  --response-manifest results/manifests/consolidated_judge.json \
  --out-dir results/behavioral_judges_v2 \
  --run-live --scope confirmatory


## 5. Recompute the confirmatory endpoints (CPU, ~1 min)

In [ ]:
import glob
judged = sorted(glob.glob('results/behavioral_judges_v2/behavioral_judges_v2_*.json'))[-1]
print('Using:', judged)
bench = sorted(glob.glob('data/frozen_v2/benchmark_v2_*.jsonl'))[-1]
get_ipython().system(
    f'python -m src.analysis.confirmatory_behavioral_endpoints '
    f'--judged {judged} --benchmark {bench} '
    f'--out results/summaries/confirmatory_endpoints.json')


---
## 6. FINAL SUMMARY - paste this whole cell's output back

This is the one block to send. Everything after this is CPU work that does
not need you.

In [ ]:
import json, os, glob
from collections import Counter

print('=' * 72)
print('FINAL GPU RUN - SUMMARY')
print('=' * 72)
print('commit:', os.popen('git rev-parse HEAD').read().strip())

print('\n--- artifacts ---')
for st in ['M3', 'M3_direct', 'M3_alt', 'M3_direct_alt']:
    for tag, exp in (('_fullAD', 900), ('_xfit5', 360)):
        p = f'results/raw/causal_ablation_v2_{st}_L24-28{tag}.json'
        if os.path.exists(p):
            n = len(json.load(open(p, encoding='utf-8', errors='replace')))
            print(f'  {st:16s}{tag:8s} {n:5d} rows {"OK" if n == exp else "<-- CHECK"}')
        else:
            print(f'  {st:16s}{tag:8s} MISSING')

print('\n--- cross-fit fold directions (cos with the full direction, L24) ---')
for st in ['M3', 'M3_direct', 'M3_alt', 'M3_direct_alt']:
    p = f'results/raw/causal_ablation_v2_{st}_L24-28_xfit5_binding.json'
    if os.path.exists(p):
        cs = [f['cos_with_full_direction_at_L24'] for f in json.load(open(p, encoding='utf-8'))['folds']]
        print(f'  {st:16s} ' + '  '.join(f'{c:.4f}' for c in cs))

print('\n--- confirmatory endpoints ---')
rep = json.load(open('results/summaries/confirmatory_endpoints.json', encoding='utf-8'))
print('status:', rep.get('status'))
cf1 = rep.get('CF1', {})
if cf1:
    print(f"CF1 Delta_C = {cf1['delta_c']:+.4f}  [{cf1['ci_low']:+.4f}, {cf1['ci_high']:+.4f}]"
          f"  n={cf1['n_effective_pairs']}")
for st, block in rep.get('CF2_by_stage', {}).items():
    print(f'\n{st}:')
    for key in ('primary', 'estimation_split_only', 'cross_fitted', 'full_A_sensitivity'):
        b = block.get(key) or {}
        n = b.get('n_effective_triples', 0)
        if n:
            print(f"  {key:22s} {b['cf2']:+.4f}  [{b['ci_low']:+.4f}, {b['ci_high']:+.4f}]  n={n}")
        else:
            print(f"  {key:22s} n=0 (not judged / not run)")

print('\n--- judge coverage ---')
judged = sorted(glob.glob('results/behavioral_judges_v2/behavioral_judges_v2_*.json'))[-1]
print('judged file:', os.path.basename(judged),
      f'({os.path.getsize(judged) / 1e6:.0f} MB)')
print('=' * 72)
print('DONE. Nothing else needs a GPU.')
print('=' * 72)
